# Copy annotated beach cell data

This notebook reads `Data/Raw/Descriptor.csv` and copies only the annotated cell matrix files from the mounted external volume into `Data/Raw/Annotated_data`, preserving date folders.

Expected source layout:

```text
/Volumes/BACKUPS/Piotr_beach/beach_all/<date>/<cell_id><feature>.txt
```

Destination layout:

```text
Data/Raw/Annotated_data/<date>/<cell_id><feature>.txt
```

Features copied per cell: `b`, `r`, `g`, `s`, `d`.

In [1]:
from pathlib import Path
import shutil

import pandas as pd
from tqdm.auto import tqdm

In [2]:
# Paths
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "Notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DESCRIPTOR_PATH = PROJECT_ROOT / "Data" / "Raw" / "Descriptor.csv"
SOURCE_ROOT = Path("/Volumes/BACKUPS/Piotr_beach/beach_all")
DESTINATION_ROOT = PROJECT_ROOT / "Data" / "Raw" / "Annotated_data"

FEATURE_SUFFIXES = ["b", "r", "g", "s", "d"]
OVERWRITE_EXISTING = False

print(f"Project root:      {PROJECT_ROOT}")
print(f"Descriptor:        {DESCRIPTOR_PATH}")
print(f"Source root:       {SOURCE_ROOT}")
print(f"Destination root:  {DESTINATION_ROOT}")

Project root:      /Users/piotr/Praca/Nauka/publikacje/Beach_classification/Beach_classification_software
Descriptor:        /Users/piotr/Praca/Nauka/publikacje/Beach_classification/Beach_classification_software/Data/Raw/Descriptor.csv
Source root:       /Volumes/BACKUPS/Piotr_beach/beach_all
Destination root:  /Users/piotr/Praca/Nauka/publikacje/Beach_classification/Beach_classification_software/Data/Raw/Annotated_data


In [3]:
if not DESCRIPTOR_PATH.exists():
    raise FileNotFoundError(f"Descriptor file not found: {DESCRIPTOR_PATH}")

if not SOURCE_ROOT.exists():
    raise FileNotFoundError(
        f"Mounted source directory not found: {SOURCE_ROOT}. "
        "Check that the external drive is mounted before running this notebook."
    )

descriptor = pd.read_csv(
    DESCRIPTOR_PATH,
    sep=";",
    header=None,
    names=["date", "cell_id", "class"],
    dtype={"date": "string", "cell_id": "string", "class": "string"},
    encoding="utf-8-sig",
)

descriptor = descriptor.dropna(subset=["date", "cell_id"]).copy()
descriptor["date"] = descriptor["date"].str.strip()
descriptor["cell_id"] = descriptor["cell_id"].str.strip()
descriptor["class"] = descriptor["class"].str.strip()

descriptor.head(), descriptor.shape

(       date cell_id class
 0  20180722  319007     2
 1  20180722  320534     2
 2  20180722  343426     2
 3  20180722  346483     2
 4  20180722  471840     2,
 (4938, 3))

In [4]:
copy_plan = []

for row in descriptor.itertuples(index=False):
    date = row.date
    cell_id = row.cell_id
    cell_class = row[2]

    for suffix in FEATURE_SUFFIXES:
        filename = f"{cell_id}{suffix}.txt"
        source_path = SOURCE_ROOT / date / filename
        destination_path = DESTINATION_ROOT / date / filename
        copy_plan.append(
            {
                "date": date,
                "cell_id": cell_id,
                "class": cell_class,
                "feature": suffix,
                "source_path": source_path,
                "destination_path": destination_path,
                "source_exists": source_path.exists(),
                "destination_exists": destination_path.exists(),
            }
        )

plan = pd.DataFrame(copy_plan)

print(f"Annotated rows in descriptor: {len(descriptor):,}")
print(f"Files expected to copy:       {len(plan):,}")
print(f"Files found in source:        {int(plan['source_exists'].sum()):,}")
print(f"Files missing in source:      {int((~plan['source_exists']).sum()):,}")
print(f"Files already in destination: {int(plan['destination_exists'].sum()):,}")

missing = plan.loc[~plan["source_exists"], ["date", "cell_id", "class", "feature", "source_path"]]
missing.head(20)

Annotated rows in descriptor: 4,938
Files expected to copy:       24,690
Files found in source:        24,690
Files missing in source:      0
Files already in destination: 0


,date,cell_id,class,feature,source_path


In [5]:
DESTINATION_ROOT.mkdir(parents=True, exist_ok=True)

results = []
for item in tqdm(copy_plan, desc="Copying annotated files"):
    source_path = item["source_path"]
    destination_path = item["destination_path"]

    result = dict(item)

    if not source_path.exists():
        result["status"] = "missing_source"
    elif destination_path.exists() and not OVERWRITE_EXISTING:
        result["status"] = "already_exists"
    else:
        destination_path.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(source_path, destination_path)
        result["status"] = "copied"

    results.append(result)

results_df = pd.DataFrame(results)
results_df["source_path"] = results_df["source_path"].astype(str)
results_df["destination_path"] = results_df["destination_path"].astype(str)

manifest_path = DESTINATION_ROOT / "copy_manifest.csv"
results_df.to_csv(manifest_path, index=False)

print(results_df["status"].value_counts(dropna=False))
print(f"Manifest written to: {manifest_path}")

Copying annotated files: 100%|██████████| 24690/24690 [01:33<00:00, 263.38it/s]


status
copied    24690
Name: count, dtype: int64
Manifest written to: /Users/piotr/Praca/Nauka/publikacje/Beach_classification/Beach_classification_software/Data/Raw/Annotated_data/copy_manifest.csv


In [6]:
results_df.loc[results_df["status"] == "missing_source", ["date", "cell_id", "class", "feature", "source_path"]].head(50)

,date,cell_id,class,feature,source_path
